# ERK-KTR Full FOV Stimulation Pipeline
This notebook showcases how to use the ERK-KTR full FOV stimulation pipeline. The pipeline is designed to simulate the full field of view (FOV) stimulation of a cells with the ERK-KTR biosensor. As it is a demo experiment, the pipeline runs on the demo hardware provided by MicroManager. 

### Import required libraries

In [2]:
import os
import time
from rtm_pymmcore.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [3]:
# from rtm_pymmcore.microscope.MMDemo import MMDemo

# mic = MMDemo()
# mic.mmc.setChannelGroup("Channel")

# for Jungfrau Microscope enable here:
from rtm_pymmcore.microscope.Jungfrau import Jungfrau

mic = Jungfrau()
mic.mmc.setChannelGroup("TTL_ERK")

FileNotFoundError: Path does not exist: E:\pertzlab_mic_configs\micromanager\Jungfrau\TiFluoroJungfrau_w_TTL_DIGITALIO.cfg

In [4]:
## Configuration options
FIRST_FRAME_STIMULATION = 10
# N_FRAMES = 340
N_FRAMES_PHASE_1 = 4
N_FRAMES_PHASE_2 = 4


SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

TIME_BETWEEN_TIMESTEPS = 5  # time in seconds between frames
TIME_PER_FOV = 3.3  # time in seconds per fov

ADD_STIM_EXPOSURE_GROUP = (
    False  # add an entry showing the last stimulation exposure for each FOV
)
REGULAR_SPACING_BETWEEN_STIMULATIONS = (
    False  # if True, the stim_timesteps will be evenly spaced
)


## Storage path for the experiment
base_path = "E:\\Alex"
experiment_name = "Test_"
path = os.path.join(base_path, experiment_name)

path_with_old_data_for_simulation = os.path.join(
    ".", "test_exp_data", "00_01_demo_imgs"
)  # path to the folder with old data for simulation


# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
# Define Channels for which Images are taken. If no power is defined, the default power of the device will be used,
# for example, see the second channel "Cy5" below. The default power is set in the GUI
channels = []
channels.append(Channel(name="miRFP", exposure=300))
channels.append(Channel(name="mScarlet3", exposure=200))

# Channel to check for the expression of the optogenetic marker, can be used if it the marker is in the same channel as the stimulation channel.
channel_optocheck = Channel(name="mCitrine", exposure=300)
optocheck_timepoints = (0, 1)  # timepoints in frames when optocheck is done


condition = ["test"]

# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24 # Example of adding multiple conditions to the dataframe. n repreats the amount of times the condition is repeated.


n_fovs_per_well = None  ## change this variable to the amount of fovs that you have per well. Set to None if you are not working with wellplate.


# Stimulation parameters for optogenetics. The stimulation will be repeated for each condition.

stim_exposures_timesteps_phase_1 = [
    {
        "ramp_pattern_name": "Sustained_before_pause",
        "stim_exposure_list": (3,) * 300,
        "stim_timestep": (2, 3, 4),
    }
]

stim_exposures_timesteps_phase_2 = [
    {
        "ramp_pattern_name": "Sustained_after_pause",
        "stim_exposure_list": (3,) * 300,
        "stim_timestep": (2, 3, 4),
    }
]


def fix_tuples_in_stim_exposure_list(
    stim_exposures_timesteps,
):
    """Convert any range or list in the stim_exposures_timesteps_before_pause to tuples."""

    for stim_exposure_timestep in stim_exposures_timesteps:
        if isinstance(stim_exposure_timestep["stim_timestep"], range):
            stim_exposure_timestep["stim_timestep"] = tuple(
                stim_exposure_timestep["stim_timestep"]
            )
        if isinstance(stim_exposure_timestep["stim_exposure_list"], range):
            stim_exposure_timestep["stim_exposure_list"] = tuple(
                stim_exposure_timestep["stim_exposure_list"]
            )
        if isinstance(stim_exposure_timestep["stim_timestep"], list):
            stim_exposure_timestep["stim_timestep"] = tuple(
                stim_exposure_timestep["stim_timestep"]
            )
        if isinstance(stim_exposure_timestep["stim_exposure_list"], list):
            stim_exposure_timestep["stim_exposure_list"] = tuple(
                stim_exposure_timestep["stim_exposure_list"]
            )


def add_stim_parameters_to_stim_exposures_timesteps(
    stim_exposures_timesteps,
):
    """Add general stimulation parameters to each stim_exposures_timesteps_before_pause dict."""
    for stim_exposure_timestep in stim_exposures_timesteps:
        stim_exposure_timestep["stim_power"] = 10
        stim_exposure_timestep["stim_channel_name"] = "CyanStim"
        stim_exposure_timestep["stim_channel_group"] = "TTL_ERK"
        stim_exposure_timestep["stim_channel_device_name"] = "Spectra"
        stim_exposure_timestep["stim_channel_power_property_name"] = "Cyan_Level"


def print_stim_exposures_timesteps(
    stim_exposures_timesteps,
):
    """Print the stim_exposures_timesteps_before_pause in a readable format."""
    for stim_exposure_timestep in stim_exposures_timesteps:
        print("Pattern Name: ", stim_exposure_timestep["ramp_pattern_name"])

        for stim_exp, stim_timestep in zip(
            stim_exposure_timestep["stim_exposure_list"],
            stim_exposure_timestep["stim_timestep"],
        ):
            print(f"{stim_exp} at {stim_timestep}")
        print("")


for stim_exposures_timesteps in [
    stim_exposures_timesteps_phase_1,
    stim_exposures_timesteps_phase_2,
]:
    fix_tuples_in_stim_exposure_list(stim_exposures_timesteps)
    add_stim_parameters_to_stim_exposures_timesteps(stim_exposures_timesteps)
    print_stim_exposures_timesteps(stim_exposures_timesteps)

## Define the Tools that you are using for the experiment
from rtm_pymmcore.stimulation.base_stimulation import StimWholeFOV
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.erk_ktr import FE_ErkKtr
from rtm_pymmcore.feature_extraction.optocheck_fe import OptoCheckFE
from rtm_pymmcore.segmentation.cellpose_v4 import CellposeV4

segmentators = [
    {
        "name": "labels",
        "class": CellposeV4(
            custom_model_path="E:\\models\\cellpose\\LifeActH2B_mixed_with_only_H2B_v1",
            min_size=100,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = StimWholeFOV()
feature_extractor = FE_ErkKtr("labels")
tracker = TrackerTrackpy(search_range=35)
optocheck = OptoCheckFE("labels", multi_timepoint=False)


from rtm_pymmcore.img_processing_pip import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
    feature_extractor_optocheck=optocheck,
)
mic.set_pipeline(pipeline=pipeline)

Pattern Name:  Sustained_before_pause
3 at 2
3 at 3
3 at 4

Pattern Name:  Sustained_after_pause
3 at 2
3 at 3
3 at 4



ModuleNotFoundError: No module named 'cellpose'

### GUI - Napari Micromanager

#### Load GUI

In [15]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\.venv\Lib\site-packages\napari_micromanager\main_window.py:51: FutureWarning: The `_dock_widgets` property is private and should not be used in any plugin code. Please use the `dock_widgets` property instead.
  if "MinMax" not in getattr(self.viewer.window, "_dock_widgets", []):


Please execute the following code block, if you would like to set a custom ROI (e.g. smaller illumination than the full field of view of the camera). Execute it after you have started the camera once in the GUI. 

In [ ]:
if mic.SET_ROI_REQUIRED:
    mic.mmc.clearROI()
    mic.mmc.setROI(mic.ROI_X, mic.ROI_Y, mic.ROI_WIDTH, mic.ROI_HEIGHT)

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [4]:
import json

# file = os.path.join(path, "fovs.json")
file = os.path.join("fovs.json")
with open(file, "r") as f:
    data_mda_fovs = json.load(f)
data_mda_fovs = data_mda_fovs
load_from_file = True

### Use FOVs to generate dataframe for acquisition

In [1]:
def _get_mda_from_file(filename):
    import json

    file = os.path.join(filename)
    with open(file, "r") as f:
        data_mda_fovs = json.load(f)
    return data_mda_fovs


def _get_mda_from_viewer(viewer):
    data_mda_fovs = viewer.window._dock_widgets["MDA"].widget().value().stage_positions
    data_mda_fovs_dict = []
    for data_mda in data_mda_fovs:
        data_mda_fovs_dict.append(data_mda.model_dump())
    data_mda_fovs = data_mda_fovs_dict
    return data_mda_fovs


def generate_fov_objects(viewer=None, filename=None):
    if filename is not None:
        data_mda_fovs = _get_mda_from_file(filename)
    elif viewer is not None:
        data_mda_fovs = _get_mda_from_viewer(viewer)
        if data_mda_fovs is None:
            assert False, "No fovs selected. Please select fovs in the MDA widget"
    else:
        assert False, "Either viewer must be provided or from_file must be True"

    fovs = []
    for i, fov in enumerate(data_mda_fovs):
        fov_object = Fov(i)
        fov_object.x = fov.get("x")
        fov_object.y = fov.get("y")
        fov_object.z = fov.get("z") if not mic.USE_ONLY_PFS else None
        fov_object.name = str(i) if fov["name"] is None else fov["name"]
        fovs.append(fov_object)
    return fovs


def generate_df_acquire(
    fovs,
    n_frames,
    time_between_timesteps,
    time_per_fov,
    channels,
    start_time=0,
    channel_optocheck=None,
    optocheck_timepoints=None,
    phase_id=None,
    phase_name=None,
    condition=None,
):
    n_fovs_simultaneously = time_between_timesteps // time_per_fov
    optocheck_timepoints = (
        optocheck_timepoints if optocheck_timepoints is not None else [n_frames - 1]
    )
    timesteps = range(n_frames)
    dfs = []
    for fov_index, fov in enumerate(fovs):
        fov_group = fov_index // n_fovs_simultaneously
        start_time_fov = fov_group * time_between_timesteps * len(timesteps)
        if condition is None or len(condition) == 0:
            condition_fov = None
        elif len(condition) == 1:
            condition_fov = condition[0]
        else:
            condition_fov = condition[fov_index]
        for timestep in timesteps:
            if phase_id is not None:
                fname = f"{str(fov_index).zfill(3)}_{str(phase_id).zfill(2)}_{str(timestep).zfill(5)}"
            else:
                fname = f"{str(fov_index).zfill(3)}_{str(timestep).zfill(5)}"
            row = {
                "fov_object": fov,
                "fov": fov_index,
                "fov_x": fov.x,
                "fov_y": fov.y,
                "fov_z": fov.z,
                "fov_name": fov.name,
                "timestep": timestep,
                "time": start_time_fov + timestep * time_between_timesteps,
                "channels": tuple(dataclasses.asdict(channel) for channel in channels),
                "fname": fname,
            }
            if condition_fov is not None:
                row["cell_line"] = condition_fov
            if channel_optocheck is not None:
                row["optocheck"] = True if timestep in optocheck_timepoints else False
                if isinstance(channel_optocheck, list):
                    row["optocheck_channels"] = tuple(
                        dataclasses.asdict(channel) for channel in channel_optocheck
                    )
                else:
                    row["optocheck_channels"] = tuple(
                        [dataclasses.asdict(channel_optocheck)]
                    )
            dfs.append(row)

    df_acquire = pd.DataFrame(dfs)
    if phase_name is not None:
        df_acquire["phase"] = phase_name
    if phase_id is not None:
        df_acquire["phase_id"] = phase_id
    print(f"Total Experiment Time: {df_acquire['time'].max()/3600}h")
    return df_acquire


def apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_exposures_timesteps,
    condition,
    n_fovs_per_well=None,
    add_stim_exposure_group=False,
    regular_spacing_between_stimulations=False,
):
    """Apply stim treatments to the df_acquire dataframe."""

    n_fovs = len(df_acquire["fov"].unique())
    n_stim_treatments = len(stim_exposures_timesteps)
    if n_stim_treatments > 0:
        n_fovs_per_stim_condition = (
            n_fovs // n_stim_treatments // len(np.unique(condition))
        )
        stim_treatment_tot = []
        random.shuffle(stim_exposures_timesteps)
        if n_fovs_per_well is None:
            for fov_index in range(0, n_fovs_per_stim_condition):
                stim_treatment_tot.extend(stim_exposures_timesteps)
            random.shuffle(stim_treatment_tot)

            if n_fovs % n_stim_treatments != 0:
                print(
                    f"Warning: Not equal number of fovs per stim condition. {n_fovs % n_stim_treatments} fovs will have repeated treatment"
                )
                stim_treatment_tot.extend(
                    stim_exposures_timesteps[: n_fovs % n_stim_treatments]
                )
            print(f"Doing {n_fovs_per_stim_condition} experiment per stim condition")

            if len(condition) != 1:
                stim_treatment_tot = stim_treatment_tot * len(np.unique(condition))

            df_acquire = pd.merge(
                df_acquire,
                pd.DataFrame(stim_treatment_tot),
                left_on="fov",
                right_index=True,
            )
        else:
            stim_treatment_tot = []
            for cell_line in np.unique(condition):
                fovs_for_one_cell_line = df_acquire.query(f"cell_line == @cell_line")[
                    "fov"
                ].unique()
                stim_treat = [
                    stim
                    for stim in stim_exposures_timesteps
                    for _ in range(n_fovs_per_well)
                ]
                if len(fovs_for_one_cell_line) != len(stim_treat):
                    print(
                        f"Warning: Number of fovs ({len(fovs_for_one_cell_line)}) for cell line {cell_line} does not match number of stim treatments ({len(stim_treat)})."
                    )
                stim_treat = pd.DataFrame(stim_treat)
                stim_treat["fov"] = fovs_for_one_cell_line
                stim_treatment_tot.append(stim_treat)
            stim_treat = pd.concat(stim_treatment_tot, ignore_index=True)
            df_acquire = pd.merge(
                df_acquire, stim_treat, left_on="fov", right_on="fov", how="left"
            )

        df_acquire["stim_exposure"] = np.nan

        for fov in df_acquire["fov"].unique():
            fov_data = df_acquire[df_acquire["fov"] == fov]

            stim_pattern = fov_data.iloc[0]

            if isinstance(stim_pattern["stim_timestep"], tuple) and isinstance(
                stim_pattern["stim_exposure_list"], tuple
            ):
                exposure_map = dict(
                    zip(
                        stim_pattern["stim_timestep"],
                        stim_pattern["stim_exposure_list"],
                    )
                )

                for timestep in fov_data["timestep"]:
                    if timestep in exposure_map:
                        mask = (df_acquire["fov"] == fov) & (
                            df_acquire["timestep"] == timestep
                        )
                        df_acquire.loc[mask, "stim_exposure"] = exposure_map[timestep]

        df_acquire["stim"] = df_acquire.apply(
            lambda row: (
                row["timestep"] in row["stim_timestep"] and row["stim_exposure"] > 0
            ),
            axis=1,
        )

    df_acquire = df_acquire.sort_values(by=["time", "fov"]).reset_index(drop=True)
    df_acquire = df_acquire.dropna(axis=1, how="all")
    if add_stim_exposure_group and regular_spacing_between_stimulations:
        spacing_interval = (
            df_acquire["stim_timestep"][0][1] - df_acquire["stim_timestep"][0][0]
        )
        for start in range(0, df_acquire["timestep"].max(), spacing_interval):
            end = start + spacing_interval
            mask = (df_acquire["timestep"] >= start) & (df_acquire["timestep"] < end)
            window = df_acquire.loc[mask, "stim_exposure"]
            value = window.dropna().iloc[0] if window.dropna().size > 0 else np.nan
            df_acquire.loc[mask, "stim_exposure"] = value

    else:
        df_acquire["stim_exposure"] = df_acquire["stim_exposure"].fillna(0)

    return df_acquire


fovs = generate_fov_objects(viewer=viewer)
fovs

df_acquire = generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_1,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    phase_name="PreDrug",
    phase_id=1,
    condition=condition,
)
df_acquire = apply_stim_treatments_to_df_acquire(
    df_acquire,
    stim_exposures_timesteps_phase_1,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire

NameError: name 'viewer' is not defined

In [38]:
df_acquire_2 = generate_df_acquire(
    fovs,
    n_frames=N_FRAMES_PHASE_2,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    channel_optocheck=channel_optocheck,
    optocheck_timepoints=optocheck_timepoints,
    phase_name="PostDrug",
    phase_id=2,
    condition=condition,
)
df_acquire_2 = apply_stim_treatments_to_df_acquire(
    df_acquire_2,
    stim_exposures_timesteps_phase_2,
    condition,
    n_fovs_per_well=n_fovs_per_well,
    add_stim_exposure_group=ADD_STIM_EXPOSURE_GROUP,
    regular_spacing_between_stimulations=REGULAR_SPACING_BETWEEN_STIMULATIONS,
)
df_acquire_2

Total Experiment Time: 0.009722222222222222h
Doing 2 experiment per stim condition


,fov_object,fov,fov_x,fov_y,fov_name,timestep,time,channels,fname,cell_line,...,ramp_pattern_name,stim_exposure_list,stim_timestep,stim_power,stim_channel_name,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim
0,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-2133.7,3348.4,0,0,0.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_02_00000,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
1,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-2133.7,3348.4,0,1,5.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_02_00001,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
2,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-2133.7,3348.4,0,2,10.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_02_00002,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,3.0,True
3,<rtm_pymmcore.data_structures.Fov object at 0x...,0,-2133.7,3348.4,0,3,15.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",000_02_00003,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,3.0,True
4,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-2133.7,3348.4,1,0,20.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_02_00000,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
5,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-2133.7,3348.4,1,1,25.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_02_00001,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,0.0,False
6,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-2133.7,3348.4,1,2,30.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_02_00002,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,3.0,True
7,<rtm_pymmcore.data_structures.Fov object at 0x...,1,-2133.7,3348.4,1,3,35.0,"({'name': 'miRFP', 'exposure': 300, 'group': N...",001_02_00003,test,...,Sustained_after_pause,"(3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","(2, 3, 4)",10,CyanStim,TTL_ERK,Spectra,Cyan_Level,3.0,True


### Run experiment

In [39]:
for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass


mic.run_experiment(df_acquire)

In [42]:
mic.run_experiment(df_acquire_2)

In [44]:
pd.read_parquet(os.path.join(path, "tracks", "000_02_00003.parquet"))

,mean_intensity_C0_nuc,mean_intensity_C1_nuc,label,x,y,area,median_intensity_C0_nuc,median_intensity_C1_nuc,mean_intensity_C0_ring,mean_intensity_C1_ring,...,stim_channel_group,stim_channel_device_name,stim_channel_power_property_name,stim_exposure,stim,optocheck,time_acquired,optocheck_channels,optocheck_mean_intensity_x,optocheck_mean_intensity_y
0,968.833333,2637.446970,2,10.795455,720.962121,528.0,975.0,2679.0,368.140940,1655.315436,...,TTL_ERK,Spectra,Cyan_Level,0.0,False,False,2025-11-05-11:02:30,None,364.269303,361.048689
1,953.706089,1376.590164,3,34.570258,122.281030,854.0,947.0,1375.0,402.815900,1575.723849,...,TTL_ERK,Spectra,Cyan_Level,0.0,False,False,2025-11-05-11:02:30,None,318.530162,316.559028
2,2954.968401,8167.429368,4,46.195167,326.133829,538.0,2908.5,8420.5,859.291457,4334.854271,...,TTL_ERK,Spectra,Cyan_Level,0.0,False,False,2025-11-05-11:02:30,None,648.962687,639.525046
3,1789.923930,5075.862124,5,66.746434,198.833597,631.0,1784.0,5297.0,435.366667,1936.285714,...,TTL_ERK,Spectra,Cyan_Level,0.0,False,False,2025-11-05-11:02:30,None,449.963665,441.935938
4,856.501471,2316.448529,6,69.055882,714.608824,680.0,861.5,2340.5,383.640187,1529.401869,...,TTL_ERK,Spectra,Cyan_Level,0.0,False,False,2025-11-05-11:02:30,None,377.448680,374.469809
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
351,3546.793220,6192.967797,93,869.311864,828.542373,590.0,3586.5,6279.0,482.292683,3818.219512,...,TTL_ERK,Spectra,Cyan_Level,3.0,True,False,2025-11-05-11:16:31,"[{'device_name': None, 'exposure': 300, 'group...",NaN,NaN
352,1801.635271,5767.735471,94,909.689379,268.190381,499.0,1818.0,5727.0,404.808511,4052.728723,...,TTL_ERK,Spectra,Cyan_Level,3.0,True,False,2025-11-05-11:16:31,"[{'device_name': None, 'exposure': 300, 'group...",NaN,NaN
353,1079.167927,3864.653555,97,975.898638,775.196672,661.0,1076.0,3990.0,377.739336,1911.635071,...,TTL_ERK,Spectra,Cyan_Level,3.0,True,False,2025-11-05-11:16:31,"[{'device_name': None, 'exposure': 300, 'group...",NaN,NaN
354,2287.658103,4486.843874,98,993.235178,896.416996,506.0,2296.0,4497.5,376.463918,3388.556701,...,TTL_ERK,Spectra,Cyan_Level,3.0,True,False,2025-11-05-11:16:31,"[{'device_name': None, 'exposure': 300, 'group...",NaN,NaN


In [ ]:
pd.read_parquet(os.path.join(path, "tracks", "000_02_00003.parquet"))

FileNotFoundError: [Errno 2] No such file or directory: 'E:\\Alex\\Test_\\tracks\\000_02_00003.parquet'

Exception in thread Thread-89 (run):
Traceback (most recent call last):
  File "C:\Users\Jungfrau\miniforge3\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\.venv\Lib\site-packages\ipykernel\ipkernel.py", line 772, in run_closure
    _threading_Thread_run(self)
  File "C:\Users\Jungfrau\miniforge3\Lib\threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\rtm_pymmcore\img_processing_pip.py", line 94, in run
    df_old = fov_obj.tracks_queue.get(block=True, timeout=360)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
_queue.Empty
Exception in thread Thread-90 (run):
Traceback (most recent call last):
  File "C:\Users\Jungfrau\miniforge3\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "c:\Users\Jungfrau\Documents\alandolt\code\rtm-pymmcore\.venv\Lib\site-packages\ipykernel\ipkernel.py"

In [ ]:
mic.post_experiment()
time.sleep(10)


fovs_i_list = os.listdir(os.path.join(path, "tracks"))
fovs_i_list.sort()
dfs = []

for fov_i in fovs_i_list:

    track_file = os.path.join(path, "tracks", fov_i)
    df = pd.read_parquet(track_file)
    dfs.append(df)

pd.concat(dfs).to_parquet(os.path.join(path, "exp_data.parquet"))

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

The following code block can be used to manually break the connection between GUI and Jupyter Notebook:


In [ ]:
### Break connection
# mm_wdg._core_link.cleanup()